# Generative Adversarial Networks

This notebook accompanies the **ML Viz** lesson on GANs.
We'll implement a simple GAN from scratch and observe the adversarial training dynamics.

**Companion lesson:** https://ml-viz.vercel.app/courses/generative-models/04-generative-adversarial-networks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## The GAN Game

A GAN has two players:
- **Generator** $G$: maps noise $z \sim \mathcal{N}(0, I)$ to fake data $\hat{x}$
- **Discriminator** $D$: tries to classify real vs fake

The minimax objective:
$$\min_G \max_D \; \mathbb{E}[\log D(x)] + \mathbb{E}[\log(1 - D(G(z)))]$$

In [ ]:
class GAN:
    """Simple GAN with 2D data."""
    
    def __init__(self, latent_dim=2, data_dim=2):
        # Generator: z -> x
        self.gW1 = np.random.randn(latent_dim, 32) * np.sqrt(2.0 / latent_dim)
        self.gb1 = np.zeros(32)
        self.gW2 = np.random.randn(32, data_dim) * np.sqrt(2.0 / 32)
        self.gb2 = np.zeros(data_dim)
        
        # Discriminator: x -> [0, 1]
        self.dW1 = np.random.randn(data_dim, 32) * np.sqrt(2.0 / data_dim)
        self.db1 = np.zeros(32)
        self.dW2 = np.random.randn(32, 1) * np.sqrt(2.0 / 32)
        self.db2 = np.zeros(1)
    
    def relu(self, x): return np.maximum(0, x)
    def sigmoid(self, x): return 1 / (1 + np.exp(-np.clip(x, -10, 10)))
    
    def generate(self, z):
        self.g_h = self.relu(z @ self.gW1 + self.gb1)
        return self.g_h @ self.gW2 + self.gb2
    
    def discriminate(self, x):
        self.d_h = self.relu(x @ self.dW1 + self.db1)
        return self.sigmoid(self.d_h @ self.dW2 + self.db2)
    
    def train_step(self, real_data, lr=0.005):
        n = real_data.shape[0]
        
        # Generate fake data
        z = np.random.randn(n, self.gW1.shape[0])
        fake_data = self.generate(z)
        
        # Discriminator forward
        d_real = self.discriminate(real_data)
        d_fake = self.discriminate(fake_data)
        
        # Discriminator loss (BCE)
        d_loss_real = -np.mean(np.log(d_real + 1e-8))
        d_loss_fake = -np.mean(np.log(1 - d_fake + 1e-8))
        d_loss = d_loss_real + d_loss_fake
        
        # Update discriminator
        d_out = np.concatenate([d_real, d_fake])
        d_labels = np.concatenate([np.ones((n, 1)), np.zeros((n, 1))])
        d_x = np.concatenate([real_data, fake_data])
        d_pred = np.concatenate([d_real, d_fake])
        d_err = d_pred - d_labels
        
        self.dW2 -= lr * self.d_h.T @ d_err / (2*n)
        self.db2 -= lr * d_err.mean(axis=0)
        d_relu = d_err @ self.dW2.T * (self.d_h > 0)
        self.dW1 -= lr * d_x.T @ d_relu / (2*n)
        self.db1 -= lr * d_relu.mean(axis=0)
        
        # Generator loss (fool discriminator)
        z = np.random.randn(n, self.gW1.shape[0])
        fake_data = self.generate(z)
        d_fake = self.discriminate(fake_data)
        g_loss = -np.mean(np.log(d_fake + 1e-8))
        
        # Update generator
        g_err = -(1 - d_fake)  # gradient of -log(D(G(z)))
        self.dW2 -= lr * self.d_h.T @ g_err / n
        self.db2 -= lr * g_err.mean(axis=0)
        d_relu2 = g_err @ self.dW2.T * (self.d_h > 0)
        self.gW2 -= lr * self.g_h.T @ d_relu2 / n
        self.gb2 -= lr * d_relu2.mean(axis=0)
        g_relu = d_relu2 @ self.gW2.T * (self.g_h > 0)
        self.gW1 -= lr * z.T @ g_relu / n
        self.gb1 -= lr * g_relu.mean(axis=0)
        
        return d_loss, g_loss

print('GAN class defined.')

## Generate target data

We'll train the GAN on a 2D mixture of Gaussians — a ring of clusters.

In [ ]:
np.random.seed(42)
n_clusters = 5
n_per = 100

centers = np.array([[2 * np.cos(2*np.pi*i/n_clusters), 2 * np.sin(2*np.pi*i/n_clusters)] for i in range(n_clusters)])

X_real = np.vstack([np.random.randn(n_per, 2) * 0.3 + c for c in centers])

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X_real[:, 0], X_real[:, 1], c='#818cf8', s=10, alpha=0.6)
ax.set_title('Real Data — 5 Gaussian Clusters', color='white', fontsize=12)
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Train the GAN

Watch the generator learn to produce the 5-cluster structure.

In [ ]:
gan = GAN(latent_dim=2, data_dim=2)

d_losses, g_losses = [], []
snapshots = []

for epoch in range(3000):
    idx = np.random.choice(len(X_real), 64, replace=False)
    d_loss, g_loss = gan.train_step(X_real[idx], lr=0.003)
    d_losses.append(d_loss)
    g_losses.append(g_loss)
    
    if epoch in [0, 100, 500, 1000, 2000, 2999]:
        z = np.random.randn(300, 2)
        samples = gan.generate(z)
        snapshots.append((epoch, samples.copy()))

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(d_losses, color='#f43f5e', alpha=0.7, linewidth=0.5)
axes[0].set_title('Discriminator Loss', color='white', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[1].plot(g_losses, color='#14b8a6', alpha=0.7, linewidth=0.5)
axes[1].set_title('Generator Loss', color='white', fontsize=12)
axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.show()

## Training snapshots

See how the generator's output evolves over training.

In [ ]:
fig, axes = plt.subplots(1, len(snapshots), figsize=(4 * len(snapshots), 4))
fig.suptitle('Generator Output Over Training', color='white', fontsize=13, y=1.02)

for ax, (epoch, samples) in zip(axes, snapshots):
    ax.scatter(samples[:, 0], samples[:, 1], c='#14b8a6', s=8, alpha=0.6)
    ax.scatter(X_real[:, 0], X_real[:, 1], c='#94a3b8', s=5, alpha=0.2)
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal')
    ax.set_title(f'Epoch {epoch}', color='white', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Mode collapse experiment

Mode collapse happens when the generator finds a few outputs that fool the discriminator
and stops exploring. Let's simulate this scenario.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Mode Collapse: Full Coverage vs Collapsed', color='white', fontsize=13, y=1.02)

# Full coverage (good)
np.random.seed(42)
good_samples = np.vstack([np.random.randn(60, 2) * 0.3 + c for c in centers])
axes[0].scatter(good_samples[:, 0], good_samples[:, 1], c='#14b8a6', s=10, alpha=0.6)
axes[0].set_title('Good: Covers all modes', color='#14b8a6', fontsize=11)

# Partial collapse
partial = np.vstack([
    np.random.randn(100, 2) * 0.3 + centers[0],
    np.random.randn(100, 2) * 0.3 + centers[1],
    np.random.randn(50, 2) * 0.2 + centers[3],  # fewer samples here
])
axes[1].scatter(partial[:, 0], partial[:, 1], c='#eab308', s=10, alpha=0.6)
axes[1].set_title('Partial: Misses some modes', color='#eab308', fontsize=11)

# Full collapse
collapsed = np.random.randn(200, 2) * 0.3 + centers[2]
axes[2].scatter(collapsed[:, 0], collapsed[:, 1], c='#f43f5e', s=10, alpha=0.6)
axes[2].set_title('Bad: Single mode collapse', color='#f43f5e', fontsize=11)

for ax in axes:
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Key takeaways

1. GANs train via a minimax game between generator and discriminator
2. The generator learns to map random noise to realistic data
3. **Mode collapse** is a common failure — the generator lacks diversity
4. Training can be unstable — both players must stay balanced
5. Despite challenges, GANs produce the sharpest samples of any generative model

**Next:** Diffusion models combine GAN-quality samples with VAE-like training stability.

## ✏️ Your turn

### Exercise 1 — Discriminator BCE loss

The discriminator is trained to maximize $\log D(x_{real}) + \log(1 - D(G(z)))$, which
is equivalent to minimizing the binary cross-entropy:

$$\mathcal{L}_D = -\mathbb{E}[\log D(x_{real})] - \mathbb{E}[\log(1 - D(G(z)))]$$

Implement it and verify: perfect discriminator gives loss 0; the total is the sum of the two terms.

In [ ]:
import numpy as np

def discriminator_loss(d_real, d_fake):
    """BCE loss for the discriminator.
    d_real: D(x) for real samples (array), d_fake: D(G(z)) for fake samples (array).
    Returns the scalar loss."""
    # TODO(you): implement the formula above
    ...

In [ ]:
d_real = np.array([0.9, 0.8, 0.7])
d_fake = np.array([0.3, 0.2, 0.4])

loss = discriminator_loss(d_real, d_fake)

assert loss > 0, "loss must be positive for imperfect discrimination"
# perfect discriminator: D(real)=1, D(fake)=0 → both log terms → 0
eps = 1e-7
perfect_loss = discriminator_loss(
    np.ones(3) - eps, np.zeros(3) + eps)
assert perfect_loss < 0.01, \
    "a near-perfect discriminator should have loss close to 0"
# loss = -mean(log d_real) - mean(log(1 - d_fake))
expected = -np.mean(np.log(d_real)) - np.mean(np.log(1 - d_fake))
assert abs(loss - expected) < 1e-12, "loss must match the formula exactly"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def discriminator_loss(d_real, d_fake):
    return -np.mean(np.log(d_real)) - np.mean(np.log(1 - d_fake))
```

</details>

### Exercise 2 — Optimal discriminator

At equilibrium the optimal discriminator for any input $x$ is:

$$D^*(x) = \frac{p_{data}(x)}{p_{data}(x) + p_g(x)}$$

This equals 0.5 when the generator perfectly matches the data distribution (Nash equilibrium).
Implement it and verify the Nash equilibrium condition.

In [ ]:
def optimal_discriminator(p_data, p_g):
    """Optimal discriminator value for a point where data density is p_data
    and generator density is p_g."""
    # TODO(you): implement D*(x) = p_data / (p_data + p_g)
    ...

In [ ]:
assert abs(optimal_discriminator(0.7, 0.3) - 0.7) < 1e-12, \
    "D*(x) = 0.7 / (0.7+0.3) = 0.7 when p_data dominates"
assert abs(optimal_discriminator(0.5, 0.5) - 0.5) < 1e-12, \
    "D*(x) = 0.5 at Nash equilibrium (p_data == p_g)"
assert abs(optimal_discriminator(0.0, 1.0) - 0.0) < 1e-12, \
    "if only generator produces this point, D*=0"
assert abs(optimal_discriminator(1.0, 0.0) - 1.0) < 1e-12, \
    "if only real data produces this point, D*=1"
# scaling both densities by k leaves D* unchanged
k = 3.14
assert abs(optimal_discriminator(k*0.6, k*0.4) - optimal_discriminator(0.6, 0.4)) < 1e-12, \
    "D* is invariant to uniform scaling of both densities"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def optimal_discriminator(p_data, p_g):
    return p_data / (p_data + p_g)
```

</details>